# Agents with create_agent

An **agent** automates the **tool-calling loop** from **12. Tools and Tool Calling**: the model decides which tools to call, LangChain executes them, results flow back as **`ToolMessage`** objects, and the cycle repeats until the model produces a final answer. In LangChain **1.x**, **`create_agent`** is the standard way to build that loop — it returns a **compiled LangGraph** that handles tool routing, parallel calls, and stopping conditions.

This notebook covers **`create_agent`** basics, **`invoke`** / **`stream`** I/O, **`system_prompt`**, **`response_format`** for structured outputs (**05**), **middleware** (call limits, retries), **checkpointing** for multi-turn threads, debugging agent runs, a **documentation agent** with search tools, and production guidance. Explicit LangGraph authoring is outside this track — here we use LangChain's agent factory.

Prerequisites: **03. Chat Models and Messages**, **05. Output Parsers and Structured Output**, **12. Tools and Tool Calling**. Persistent chat memory patterns continue in **14. Memory and Chat History**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain
import langchain_core

print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)

---

## 1. From Manual Loop to Agent

**12. Tools and Tool Calling** built `run_tool_loop` manually:

```python
messages = [HumanMessage(...)]
for step in range(max_steps):
    ai = llm.bind_tools(tools).invoke(messages)
    messages.append(ai)
    if not ai.tool_calls:
        return ai.content
    messages.extend(execute_tool_calls(ai))
```

**`create_agent`** replaces that skeleton with a **state graph**:

```
User messages ──► model node ──► tool_calls?
                      ▲               │
                      │               ▼ yes
                      └── tool node ◄─┘
                              │
                              ▼ no tool_calls
                         final answer
```

| Concern | Manual loop | `create_agent` |
|---------|-------------|----------------|
| Tool execution | You write `execute_tool_calls` | Built-in tool node |
| Parallel tool calls | You must handle all `tool_call_id`s | Built-in |
| Iteration cap | Your `max_steps` | Middleware + graph limits |
| Checkpointing | You persist `messages` | `checkpointer` + `thread_id` |
| Structured final output | Custom parsing | `response_format=` |

Keep the manual loop mental model — it is exactly what the agent graph executes.

---

## 2. LangChain 1.x — What Replaced AgentExecutor

Older tutorials used **`create_tool_calling_agent`** + **`AgentExecutor`**. LangChain 1.x consolidates on:

| Legacy (0.x docs) | LangChain 1.x |
|-------------------|---------------|
| `AgentExecutor` | **`create_agent`** graph |
| `AgentExecutor(max_iterations=…)` | **`ModelCallLimitMiddleware`** / **`ToolCallLimitMiddleware`** |
| Custom agent class | **`middleware=`** hooks |
| Memory via `ConversationBufferMemory` | **`checkpointer`** + message state (**14**) |

`create_agent` is implemented with **LangGraph** under the hood (**01. Introduction to LangChain**) but exposes a simple factory API — you do not need to draw nodes and edges for standard tool agents.

---

## 3. Setup — Model, Data, and Tools

Same **Notes API** lesson data and tools as **12**:

In [ ]:
from langchain_core.tools import StructuredTool, tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
]


def _search_notes(query: str) -> str:
    q = query.lower()
    hits = [c for c in DOC_CHUNKS if q in c["text"].lower() or q in c["id"]]
    if not hits:
        return "No matching notes."
    return "\n".join(f"[{c['id']}] {c['text']}" for c in hits)


@tool
def get_note(note_id: str) -> str:
    """Fetch a note by id (n1, n2, n3, n4)."""
    return NOTES.get(note_id, f"Unknown note id: {note_id}")


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


search_notes = StructuredTool.from_function(
    func=_search_notes,
    name="search_notes",
    description="Search internal documentation chunks by keyword.",
)


class ListNotesInput(BaseModel):
    prefix: str = Field(description="Note id prefix filter, e.g. 'n' for all notes")
    limit: int = Field(default=10, ge=1, le=50, description="Max notes to return")


@tool(args_schema=ListNotesInput)
def list_notes(prefix: str, limit: int = 10) -> str:
    """List note ids and previews filtered by id prefix."""
    items = [(k, v) for k, v in NOTES.items() if k.startswith(prefix)][:limit]
    return "\n".join(f"{k}: {v[:60]}..." for k, v in items) or "No notes matched."


tools = [get_note, search_notes, add, list_notes]
print("tools:", [t.name for t in tools])

---

## 4. create_agent — Minimal Agent

**`create_agent(model, tools, system_prompt=...)`** returns a **`CompiledStateGraph`**:

In [ ]:
from langchain.agents import create_agent

SYSTEM = """You are a Notes API assistant.
Use tools to look up documentation before answering factual questions.
Prefer search_notes for open questions and get_note when you know the note id.
For arithmetic, use the add tool.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM,
)

print(type(agent).__name__)
print("graph nodes:", list(agent.get_graph().nodes))

### 4.1 Key Parameters

| Parameter | Purpose |
|-----------|--------|
| **`model`** | `ChatOpenAI` instance or provider string (`"openai:gpt-4o-mini"`) |
| **`tools`** | List of `@tool` / `StructuredTool` callables |
| **`system_prompt`** | Default system instructions for every run |
| **`response_format`** | Pydantic model or schema for typed final output (**05**) |
| **`middleware`** | Hooks: limits, retries, summarization, HITL |
| **`checkpointer`** | Persist graph state across turns (**§9**, **14**) |
| **`interrupt_before` / `interrupt_after`** | Human-in-the-loop pause points |
| **`debug`** | Verbose logging during development |

---

## 5. Invoking an Agent

Input is a **state dict** with a **`messages`** list. Output includes the full **message trace** plus optional **`structured_response`**:

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage


def last_ai_content(result: dict) -> str:
    """Extract the final assistant text from an agent result."""
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            return msg.content if isinstance(msg.content, str) else str(msg.content)
    return ""


result = agent.invoke({
    "messages": [HumanMessage(content="What pytest file holds DB fixtures? Use tools.")],
})

print("answer:", last_ai_content(result))
print("message count:", len(result["messages"]))

### 5.1 Reading the Message Trace

Inspect intermediate steps for debugging:

In [ ]:
for i, msg in enumerate(result["messages"]):
    role = msg.type
    if isinstance(msg, AIMessage) and msg.tool_calls:
        names = [tc["name"] for tc in msg.tool_calls]
        print(f"{i:02d} {role:6s} tool_calls={names}")
    elif isinstance(msg, ToolMessage):
        preview = str(msg.content)[:70].replace("\n", " ")
        print(f"{i:02d} {role:6s} {preview}...")
    else:
        preview = str(msg.content)[:70].replace("\n", " ")
        print(f"{i:02d} {role:6s} {preview}...")

Typical trace: **`human` → `ai` (tool_calls) → `tool` → `ai` (final text)**. Multi-tool and multi-step runs extend this pattern.

---

## 6. Multi-Step and Multi-Tool Runs

Agents may call **several tools across multiple turns** before answering:

In [ ]:
multi = agent.invoke({
    "messages": [
        HumanMessage(
            content=(
                "Search for JWT documentation, fetch note n3, "
                "and explain the auth header format."
            )
        )
    ],
})

tool_steps = [
    m for m in multi["messages"]
    if isinstance(m, (ToolMessage, AIMessage)) and (
        isinstance(m, ToolMessage) or m.tool_calls
    )
]
print(f"tool-related messages: {len(tool_steps)}")
print("answer:", last_ai_content(multi))

In [ ]:
math_result = agent.invoke({
    "messages": [HumanMessage(content="What is 17 plus 28? Use the add tool.")],
})
print(last_ai_content(math_result))

---

## 7. Structured Agent Output — response_format

When APIs need **typed** responses, pass a Pydantic model to **`response_format`** (**05. Output Parsers and Structured Output**):

In [ ]:
class AgentAnswer(BaseModel):
    answer: str = Field(description="User-facing answer grounded in tool results")
    sources: list[str] = Field(description="Note ids or chunk ids consulted, e.g. n3, c2")
    used_tools: list[str] = Field(description="Tool names invoked during the run")


structured_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM + "\nPopulate sources with note/chunk ids you relied on.",
    response_format=AgentAnswer,
)

structured_result = structured_agent.invoke({
    "messages": [HumanMessage(content="What header carries JWT? Use tools.")],
})

typed = structured_result["structured_response"]
print(type(typed).__name__)
print(typed)

**`structured_response`** is populated when the agent finishes. Prose answers still appear in **`messages`** — use the typed object for downstream validation and storage.

---

## 8. Streaming Agent Runs

Agents support **`stream`** and **`astream`** like other LangGraph graphs (**07. Streaming, Batch, and Async**):

In [ ]:
print("Streaming final answer tokens (stream_mode='messages'):")
for event in agent.stream(
    {"messages": [HumanMessage(content="List the first two notes. Use list_notes.")]},
    stream_mode="messages",
):
    token, metadata = event
    if token.content and metadata.get("langgraph_node") == "model":
        print(token.content, end="", flush=True)
print("\n--- done ---")

### 8.1 Stream Modes

| `stream_mode` | Yields | Use when |
|---------------|--------|----------|
| **`"messages"`** | Message/token chunks | Token-by-token UI |
| **`"updates"`** | Per-node state deltas | Step progress indicators |
| **`"values"`** | Full state after each step | Debugging full message list |

Tool execution completes before the model streams the next turn — show "Running tool…" during tool nodes.

---

## 9. Checkpointing and thread_id

Pass a **`checkpointer`** to persist conversation state. Each session uses a **`thread_id`** in **`config`** — preview of full memory patterns in **14. Memory and Chat History**:

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
memory_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM,
    checkpointer=checkpointer,
)

thread_config = {"configurable": {"thread_id": "demo-thread-1"}}

turn1 = memory_agent.invoke(
    {"messages": [HumanMessage(content="Tell me about Alembic migrations in the notes.")]},
    config=thread_config,
)
print("turn 1:", last_ai_content(turn1)[:120], "...")

turn2 = memory_agent.invoke(
    {"messages": [HumanMessage(content="What command do I run?")]},
    config=thread_config,
)
print("turn 2:", last_ai_content(turn2))

The checkpointer stores prior **`messages`** for the thread — the agent can resolve follow-ups without you manually concatenating history. Production apps use durable checkpointers (Postgres, Redis) instead of **`MemorySaver`**.

---

## 10. Middleware — Limits and Guardrails

**`middleware`** intercepts model and tool steps. Common built-ins:

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware

guarded_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM,
    middleware=[
        ModelCallLimitMiddleware(run_limit=8, exit_behavior="end"),
        ToolCallLimitMiddleware(run_limit=6, exit_behavior="continue"),
    ],
)

guarded_result = guarded_agent.invoke({
    "messages": [HumanMessage(content="Search jwt and list all notes starting with n.")],
})
print(last_ai_content(guarded_result))

| Middleware | Purpose |
|------------|--------|
| **`ModelCallLimitMiddleware`** | Cap LLM turns per run (cost control) |
| **`ToolCallLimitMiddleware`** | Cap tool executions per run |
| **`ModelRetryMiddleware`** | Retry transient model errors |
| **`ToolRetryMiddleware`** | Retry flaky tool I/O |
| **`ModelFallbackMiddleware`** | Swap models on failure (**16**) |
| **`SummarizationMiddleware`** | Trim long histories (**14**) |
| **`HumanInTheLoopMiddleware`** | Approve tool calls before execution |

Replace your manual **`max_steps`** from **12** with middleware limits in production agents.

---

## 11. Provider-Agnostic Model Strings

**`create_agent`** accepts a model **string** resolved via **`init_chat_model`** (**03**):

In [ ]:
string_agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[add],
    system_prompt="You are a calculator assistant. Use tools for arithmetic.",
)

print(last_ai_content(string_agent.invoke({
    "messages": [HumanMessage(content="What is 9 plus 14? Use add.")],
})))

String identifiers simplify config-driven deployments — swap `"openai:gpt-4o-mini"` for `"anthropic:claude-sonnet-4-6"` without rewriting the agent factory.

---

## 12. Documentation Agent — RAG as a Tool Choice

From **12**: search can be a **tool** the agent invokes optionally — unlike **11. RAG with LCEL**, which always retrieves:

In [ ]:
@tool
def search_documentation(query: str) -> str:
    """Search internal documentation. Use for API, database, security, or testing questions."""
    return _search_notes(query)


doc_agent = create_agent(
    model=llm,
    tools=[search_documentation, get_note, add],
    system_prompt=(
        "You help engineers with the Notes API. "
        "Search documentation before answering product questions. "
        "Use add only for explicit arithmetic."
    ),
)

doc_answer = doc_agent.invoke({
    "messages": [
        HumanMessage(content="Compare FastAPI routing with JWT auth based on the docs."),
    ],
})
print(last_ai_content(doc_answer))

In [ ]:
# Pure arithmetic — agent should skip search
quick = doc_agent.invoke({
    "messages": [HumanMessage(content="What is 6 plus 7?")],
})
print(last_ai_content(quick))

| Pattern | Retrieve every query? | Model chooses tools? |
|---------|----------------------|----------------------|
| **RAG chain** (**11**) | Yes | No |
| **Agent + search tool** | Only when needed | Yes |

Many production assistants combine both: a RAG tool inside a larger agent toolbox.

---

## 13. batch and async

Agents inherit LangGraph **`batch`** / **`ainvoke`** for eval workloads (**07**):

In [ ]:
eval_inputs = [
    {"messages": [HumanMessage(content="JWT header format? Use tools.")]},
    {"messages": [HumanMessage(content="Alembic upgrade command? Use tools.")]},
    {"messages": [HumanMessage(content="What is 3 plus 4? Use add.")]},
]

batch_results = agent.batch(eval_inputs, config={"max_concurrency": 2})
for item, res in zip(eval_inputs, batch_results):
    q = item["messages"][0].content[:40]
    print(f"Q: {q}...")
    print(f"A: {last_ai_content(res)[:90]}...\n")

In [ ]:
import asyncio


async def async_agent_demo():
    res = await agent.ainvoke({
        "messages": [HumanMessage(content="Which note mentions FastAPI? Use tools.")],
    })
    print(last_ai_content(res))


asyncio.run(async_agent_demo())

---

## 14. Debugging Agent Runs

### 14.1 debug=True

Enable verbose graph logging during development:

In [ ]:
debug_agent = create_agent(
    model=llm,
    tools=[get_note, add],
    system_prompt="Debug demo agent.",
    debug=True,
)

# Uncomment to see verbose graph logs:
# debug_agent.invoke({"messages": [HumanMessage(content="get note n1")]})

### 14.2 stream_mode="values" for Step Inspection

Watch state evolve after each node:

In [ ]:
for step_i, state in enumerate(agent.stream(
    {"messages": [HumanMessage(content="Fetch note n2 and summarize.")]},
    stream_mode="values",
)):
    print(f"step {step_i}: {len(state['messages'])} messages")

Deeper tracing hooks live in **15. Callbacks, Caching, and Observability** (LangSmith, `astream_events`).

---

## 15. When to Use Agents vs Chains

| Scenario | Prefer |
|----------|--------|
| Fixed retrieve → answer | **RAG chain** (**11**) |
| Single known tool, one step | **Manual loop** or thin wrapper |
| Model picks among several tools | **`create_agent`** |
| Multi-step reasoning with APIs | **`create_agent`** |
| Strict latency / cost budget | Chain (fewer model calls) |
| Human approval before side effects | Agent + **`HumanInTheLoopMiddleware`** |

Agents trade **flexibility** for **unpredictability** — more model calls, harder eval. Start with chains; add agents when tool routing is genuinely dynamic.

---

## 16. Production Checklist

| Area | Recommendation |
|------|----------------|
| **Tools** | Small, well-described set (**12**) |
| **Limits** | `ModelCallLimitMiddleware` + `ToolCallLimitMiddleware` |
| **Memory** | `checkpointer` + per-user `thread_id` (**14**) |
| **Output** | `response_format` for API contracts (**05**) |
| **Errors** | `ToolRetryMiddleware`; tools return explicit error strings |
| **Observability** | LangSmith / callbacks (**15**) |
| **Fallbacks** | `ModelFallbackMiddleware` (**16**) |
| **Side effects** | HITL middleware or separate confirmation tool |
| **Testing** | Fake tools + assert message trace shape (**12**) |

---

## 17. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Passing bare string to `invoke` | Type/validation error | Wrap in `{"messages": [HumanMessage(...)]}` |
| Reading only last message blindly | Miss structured output | Check `structured_response` when set |
| No call limits | Runaway cost | Middleware limits |
| Too many tools | Wrong tool selection | Split agents or use tool selector middleware |
| Expecting memory without checkpointer | Forgets prior turns | Add `checkpointer` + `thread_id` |
| Using legacy `AgentExecutor` docs | Import errors | `create_agent` only in 1.x |
| Ignoring tool trace on failures | Cannot debug routing | Log full `result["messages"]` |
| RAG chain when search is optional | Wasted retrieval | Expose search as agent tool |

---

## 18. Summary

**`create_agent`** is LangChain 1.x's standard tool agent: pass a **model**, **tools**, and optional **system_prompt**, **response_format**, **middleware**, and **checkpointer**. It returns a LangGraph you invoke with **`{"messages": [...]}`** and read **`messages`** / **`structured_response`** from the result.

Key takeaways:

- Agents automate the **tool loop** from **12** — same message types, less boilerplate.
- **`last_ai_content`** / message trace inspection are essential for debugging.
- **`response_format`** delivers typed final outputs for APIs.
- **`stream_mode`** controls token vs step streaming.
- **`checkpointer` + `thread_id`** enable multi-turn threads (**14**).
- **Middleware** replaces manual iteration caps and adds retries, fallbacks, HITL.
- **Search-as-tool** lets agents choose retrieval vs direct answers.

Demonstrations built a Notes API agent, inspected message traces, streamed tokens, added structured output, checkpointed a two-turn thread, applied call-limit middleware, and compared agent vs RAG patterns.

Next: **14. Memory and Chat History** — persisting, trimming, and injecting conversation context across agent and chain runs.